# Notebook 05: Honest Evaluation**Goal:** Understand what the model gets wrong, where it biases, and what the residuals tell us about real-world behavior.This is regression — we analyze residuals, not confusion matrices.But the principle is the same: understand your errors.

In [ ]:
import pandas as pdimport numpy as npimport syssys.path.insert(0, "../src")from housing.data.load_data import load_configfrom housing.models.train_model import build_pipelinefrom housing.models.predict_model import load_modelfrom housing.visualization.visualize import plot_residual_analysis, plot_feature_importancefrom housing.features.build_features import get_feature_columnsimport matplotlib.pyplot as pltconfig = load_config("../config/config.yaml")df = pd.read_csv("../data/processed/housing_clean.csv")from sklearn.model_selection import train_test_splitnumeric_features, categorical_features, target = get_feature_columns(config)feature_cols = numeric_features + categorical_featuresX = df[feature_cols].copy()y = df[target].copy()X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.2, random_state=42, stratify=X["location"])# Load best model and predictpipeline = load_model("../models/gradient_boosting_latest.pkl")y_pred = pipeline.predict(X_test)

## 1. Residual Analysis

In [ ]:
fig = plot_residual_analysis(    y_test.values, y_pred,     locations=X_test["location"].values,    save_path="../reports/figures/residual_analysis.png")plt.show()

## 2. Feature ImportanceWhat is the model actually using to make decisions?

In [ ]:
# Get feature names after preprocessingpreprocessor = pipeline.named_steps["preprocessor"]feature_names = (    numeric_features +     list(preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_features)))fig = plot_feature_importance(    pipeline, feature_names,    save_path="../reports/figures/feature_importance.png")plt.show()

## 3. Worst PredictionsWhat are the biggest misses, and what do they tell us?

In [ ]:
residual_df = pd.DataFrame({    "location": X_test["location"].values,    "actual": y_test.values,    "predicted": y_pred,    "residual": y_test.values - y_pred,    "abs_error": np.abs(y_test.values - y_pred),    "pct_error": np.abs(y_test.values - y_pred) / y_test.values * 100})print("=== TOP 10 WORST PREDICTIONS (by absolute error) ===")worst = residual_df.nlargest(10, "abs_error")print(worst.to_string(index=False))

## 4. Bias by LocationIs the model systematically over- or under-predicting in certain areas?

In [ ]:
print("\n=== BIAS BY LOCATION ===")for loc in residual_df["location"].unique():    loc_resid = residual_df[residual_df["location"] == loc]["residual"]    print(f"{loc:>10s}: Mean residual = ${loc_resid.mean():>8,.0f}")

**Interpretation:**- **Positive residual** = model UNDER-predicted (actual > predicted)- **Negative residual** = model OVER-predicted (actual < predicted)| Location | Bias | What it means ||----------|------|---------------|| Downtown | +$7,460 | Model slightly under-predicts Downtown prices || Suburbs  | -$9,595 | Model slightly over-predicts Suburbs prices || Rural    | -$1,084 | Nearly unbiased |The Downtown premium may be even higher than the model captures, suggesting unmeasured amenities (transit, walkability, views).

## 5. Error by Price RangeWhere does the model struggle most?

In [ ]:
residual_df["price_range"] = pd.cut(    residual_df["actual"],    bins=[0, 250000, 400000, 550000, 900000],    labels=["<$250K", "$250-400K", "$400-550K", ">$550K"])range_stats = residual_df.groupby("price_range").agg({    "abs_error": ["mean", "max"],    "pct_error": "mean",    "actual": "count"}).round(1)range_stats.columns = ["Mean Abs Error", "Max Error", "Mean % Error", "Count"]print(range_stats)

**Finding:**- **Cheap homes (<$250K):** Hardest to predict (9.6% error). Often distressed/unique sales.- **Mid-range ($250-550K):** Sweet spot (3.5-5.0% error).- **Expensive (>$550K):** Low % error but high absolute dollars ($23K avg).## Honest Summary| Metric | Value | Interpretation ||--------|-------|----------------|| Test R² | 0.979 | Explains 97.9% of price variance || Test MAE | $17,655 | Average prediction is off by ~$18K || Test RMSE | $22,635 | Typical error magnitude || Best at | $250-550K homes | Most common, most predictable segment || Worst at | <$250K homes | High variance, unique properties || Bias | Location-dependent | Under-predicts Downtown, over-predicts Suburbs |**What would improve this:**- Lot size (especially for Rural/Suburbs)- School district ratings- Renovation year / condition- Distance to transit/city center